# P1 · Matched-seed TCN comparison
Runs four new jobs in sequence: baseline and balanced for each of two additional seeds. Original seed results are read from the verified reports. Enable Internet and GPU. Save the executed notebook and download the ZIP.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)], check=True)
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e","."],check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve
raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not (cache / "metadata.json").exists():
    subprocess.run([sys.executable, '-m', 'acr.data.billboard.prepare'], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
assert torch.cuda.is_available(), "Enable a GPU before the paired experiment."
repeat_root = project / "runs/p1-seed-repeat"
print((project / "configs/p1/p1-seeds.json").read_text())
subprocess.run([sys.executable, '-m', 'acr.experiments.billboard.repeat_frame',
                "--output", str(repeat_root), "--device", "cuda"], check=True)
print((repeat_root / "per_seed.csv").read_text())
print((repeat_root / "summary.json").read_text())


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/p1-seed-repeat-results", "zip",
                              project, "runs/p1-seed-repeat")
print(archive)
